# Medical History Extraction
## Identifying Pre-Existing Conditions and Reproductive History

Extract medical history and comorbidities from condition_occurrence table to create binary indicator variables for pre-existing conditions.

**Time windows:**
- **Pre-pregnancy comorbidities:** 2 years prior to LMP through LMP
- **Early pregnancy conditions:** LMP through 20 weeks gestation


File is commented out because it builds a dataset in the All of Us open research access dataset. If you have an account, you can uncomment the code to run and duplicate our cohort. 

# Setup 

In [ ]:
'''
from IPython.display import display, HTML
import pandas as pd
import numpy, scipy
import matplotlib, matplotlib.pyplot as plt
from matplotlib import rcParams
from datetime import date
import seaborn as sns
import os
import subprocess
import pickle
import numpy as np

my_bucket = os.getenv('WORKSPACE_BUCKET')
data_subfolder = 'pregnancy_cohort_1'
print(my_bucket)
'''

In [ ]:
'''
version = %env WORKSPACE_CDR
version
'''

# Load Base Pregnancy Cohort

In [ ]:
'''
base_cohort = pd.read_pickle("./pregnancy_episodes_final_7-8.pkl")
base_cohort
'''

In [ ]:
'''
print(f"Loaded cohort: {len(base_cohort):,} pregnancy episodes")
print(f"Unique individuals: {base_cohort['person_id'].nunique():,}")

# Prepare cohort with just ID and dates for merging
cohort_dates = base_cohort[['person_id', 'censor_date', 'id']].copy()

# Initialize medical history dataframe
medical_history = cohort_dates.copy()

print(f"\nInitialized medical history dataframe: {medical_history.shape}")
'''

# Medical History Conditions 

All condition concepts we're extracting with descriptions.

In [ ]:
'''
# Pre-pregnancy comorbidities (2 years before LMP to LMP)
PREPREG_CONDITIONS = {
    'dm': {'concept_id': '201820', 'name': 'Diabetes mellitus'},
    'chest_pain': {'concept_id': '77670', 'name': 'Chest pain'},
    'uti': {'concept_id': '81902', 'name': 'Urinary tract infection'},
    'hyperkeratosis': {'concept_id': '141932', 'name': 'Hyperkeratosis'},
    'ab_pain': {'concept_id': '200219, 197988', 'name': 'Abdominal pain'},
    'carp_tunnel': {'concept_id': '380094', 'name': 'Carpal tunnel syndrome'},
    'hypercholes': {'concept_id': '437827', 'name': 'Hypercholesterolemia'},
    'cyst_ovary': {'concept_id': '197610', 'name': 'Ovarian cyst'},
    'acne': {'concept_id': '141095', 'name': 'Acne'},
    'headache': {'concept_id': '378253', 'name': 'Headache'},
    'pain_limb': {'concept_id': '138525', 'name': 'Limb pain'},
    'ten_headache': {'concept_id': '376382', 'name': 'Tension headache'},
    'pain_joint': {'concept_id': '78232, 77074', 'name': 'Joint pain'},
    'pain_neck': {'concept_id': '24134', 'name': 'Neck pain'},
    'viraldisease': {'concept_id': '440029', 'name': 'Viral disease'},
    'pain_rightquad': {'concept_id': '193322', 'name': 'Right upper quadrant pain'},
    'pain_chronic': {'concept_id': '436096', 'name': 'Chronic pain'},
    'resp_infection': {'concept_id': '257011', 'name': 'Respiratory infection'},
    'breast_lump': {'concept_id': '80767', 'name': 'Breast lump'},
    'wart': {'concept_id': '140641', 'name': 'Wart'},
    'viraldisease_preg': {'concept_id': '435604', 'name': 'Viral disease in pregnancy'}
}

# Reproductive history (any time before LMP)
REPRODUCTIVE_HISTORY = {
    'hxpe_e': {'concept_id': '439393', 'name': 'History of preeclampsia/eclampsia'},
    'hxghtn': {'concept_id': '4167493', 'name': 'History of hypertension'},
    'hxgdm': {'concept_id': '4024659', 'name': 'History of gestational diabetes'},
    'hxabrt': {'concept_id': '4067106', 'name': 'History of abortion'},
    'cancer_prepreg': {'concept_id': '443392', 'name': 'Cancer before pregnancy'},
    'chtn_prepreg': {'concept_id': '316866', 'name': 'Chronic hypertension'},
    'dep_priorlmp': {'concept_id': '440383', 'name': 'Depression prior to pregnancy'},
    'pcos': {'concept_id': '40443308', 'name': 'Polycystic ovary syndrome'}
}

# Early pregnancy conditions (LMP to 20 weeks)
EARLY_PREG_CONDITIONS = {
    'preg_single': {'concept_id': '432969', 'name': 'Singleton pregnancy'},
    'chtn_earlypreg': {'concept_id': '316866', 'name': 'Hypertension in early pregnancy'}
}

print(f"Pre-pregnancy conditions defined: {len(PREPREG_CONDITIONS)}")
print(f"Reproductive history conditions defined: {len(REPRODUCTIVE_HISTORY)}")
print(f"Early pregnancy conditions defined: {len(EARLY_PREG_CONDITIONS)}")
'''

# Helper Functions

In [ ]:
'''
def custom_sql(person_ids, sql_command, num_splits=5):
    """Split large queries to avoid BigQuery timeouts"""
    CDR = os.environ['WORKSPACE_CDR']
    div_index = len(person_ids) // num_splits

    for i in range(num_splits + 1): 
        ids_string = ', '.join(str(e) for e in person_ids[i*div_index:(i + 1)*div_index])
        if len(ids_string) == 0:
            continue
        
        df_temp = pd.read_gbq(
            sql_command.format(CDR, ids_string),
            use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
            dialect="standard", 
            progress_bar_type="tqdm_notebook"
        )

        if i == 0: 
            df = df_temp 
        else: 
            df = pd.concat([df, df_temp])
    return df

def create_condition_query(concept_ids):
    """Generate SQL query for condition extraction"""
    sql = """
    SELECT 
        c_occurrence.PERSON_ID,
        c_occurrence.CONDITION_END_DATETIME,
        c_occurrence.CONDITION_CONCEPT_ID,
        c_occurrence.CONDITION_START_DATETIME,
        c_standard_concept.concept_name as STANDARD_CONCEPT_NAME
    FROM 
        (SELECT * 
        FROM `{0}.condition_occurrence` c_occurrence 
        WHERE 
            person_id IN ({1})
            AND condition_concept_id in  (
                select distinct c.concept_id 
                from `{0}.cb_criteria` c 
                join
                    (
                        select cast(cr.id as string) as id 
                        from `""" + os.environ["WORKSPACE_CDR"] + """`.cb_criteria cr 
                        where
                            domain_id = 'CONDITION' 
                            and is_standard = 1 
                            and concept_id in (""" + concept_ids + """) 
                            and is_selectable = 1 
                            and full_text like '%[condition_rank1]%'
                    ) a 
                        on (
                            c.path like concat('%.',a.id,'.%') 
                            or c.path like concat('%.',a.id) 
                            or c.path like concat(a.id,'.%') 
                            or c.path = a.id
                        ) 
                    where
                        domain_id = 'CONDITION' 
                        and is_standard = 1 
                        and is_selectable = 1
                )
        ) c_occurrence
    LEFT JOIN
        `{0}.concept` c_standard_concept 
            on c_occurrence.CONDITION_CONCEPT_ID = c_standard_concept.CONCEPT_ID
    """
    return sql

print("Helper functions defined")
'''

# Extract Pre-Pregnancy Comorbidities

Query conditions occurring 2 years before LMP through LMP.

In [ ]:
'''
person_ids = list(set(medical_history['person_id']))

print("Extracting pre-pregnancy comorbidities...")
print(f"Total conditions to query: {len(PREPREG_CONDITIONS)}\n")

for condition_name, condition_info in PREPREG_CONDITIONS.items():
    print(f"Processing: {condition_name} ({condition_info['name']})")
    
    # Create SQL query
    sql = create_condition_query(condition_info['concept_id'])
    
    # Execute query
    condition_records = custom_sql(person_ids, sql, num_splits=3)
    
    # Initialize column
    medical_history[condition_name] = 0.0
    
    # Mark pregnancies where condition occurred before LMP (within 2 years)
    for idx, pregnancy in medical_history.iterrows():
        patient_id = pregnancy['person_id']
        lmp_date = pregnancy['censor_date']
        lookback_date = lmp_date - pd.DateOffset(years=2)
        
        # Check if this person has the condition in the relevant time window
        patient_conditions = condition_records[
            (condition_records['PERSON_ID'] == patient_id) & 
            (condition_records['CONDITION_START_DATETIME'] >= lookback_date) &
            (condition_records['CONDITION_START_DATETIME'] < lmp_date)
        ]
        
        if not patient_conditions.empty:
            medical_history.at[idx, condition_name] = 1.0
    
    n_positive = medical_history[condition_name].sum()
    pct = n_positive / len(medical_history) * 100
    print(f"  Found in {n_positive:,} pregnancies ({pct:.1f}%)\n")

print("Pre-pregnancy comorbidities extraction complete")
'''

# Extract Reproductive History

In [ ]:
'''
print("Extracting reproductive history...")
print(f"Total conditions to query: {len(REPRODUCTIVE_HISTORY)}\n")

for condition_name, condition_info in REPRODUCTIVE_HISTORY.items():
    print(f"Processing: {condition_name} ({condition_info['name']})")
    
    sql = create_condition_query(condition_info['concept_id'])
    condition_records = custom_sql(person_ids, sql, num_splits=3)
    
    medical_history[condition_name] = 0.0
    
    # Mark pregnancies where condition occurred any time before LMP
    for idx, pregnancy in medical_history.iterrows():
        patient_id = pregnancy['person_id']
        lmp_date = pregnancy['censor_date']
        
        patient_conditions = condition_records[
            (condition_records['PERSON_ID'] == patient_id) & 
            (condition_records['CONDITION_START_DATETIME'] < lmp_date)
        ]
        
        if not patient_conditions.empty:
            medical_history.at[idx, condition_name] = 1.0
    
    n_positive = medical_history[condition_name].sum()
    pct = n_positive / len(medical_history) * 100
    print(f"  Found in {n_positive:,} pregnancies ({pct:.1f}%)\n")

print("Reproductive history extraction complete")
'''

# Early Pregnancy Conditions

In [ ]:
'''
print("Extracting early pregnancy conditions...\n")

# Singleton pregnancy (default = 1, mark as 0 if multiple gestation found)
print("Processing: preg_single (Singleton pregnancy)")
sql = create_condition_query('432969')  # Multiple gestation concept
multiple_gestation = custom_sql(person_ids, sql, num_splits=3)

medical_history['preg_single'] = 1.0

for idx, pregnancy in medical_history.iterrows():
    patient_id = pregnancy['person_id']
    lmp_date = pregnancy['censor_date']
    delivery_date = lmp_date + pd.DateOffset(weeks=40)
    
    # Check for multiple gestation diagnosis during pregnancy
    patient_conditions = multiple_gestation[
        (multiple_gestation['PERSON_ID'] == patient_id) & 
        (multiple_gestation['CONDITION_START_DATETIME'] > lmp_date) &
        (multiple_gestation['CONDITION_START_DATETIME'] < delivery_date)
    ]
    
    if not patient_conditions.empty:
        medical_history.at[idx, 'preg_single'] = 0.0

n_singleton = medical_history['preg_single'].sum()
pct = n_singleton / len(medical_history) * 100
print(f"  Singleton pregnancies: {n_singleton:,} ({pct:.1f}%)\n")

# Early hypertension (LMP to 20 weeks)
print("Processing: chtn_earlypreg (Hypertension in early pregnancy)")
sql = create_condition_query('316866')  # Hypertension concept
htn_records = custom_sql(person_ids, sql, num_splits=3)

medical_history['chtn_earlypreg'] = 0.0

for idx, pregnancy in medical_history.iterrows():
    patient_id = pregnancy['person_id']
    lmp_date = pregnancy['censor_date']
    week_20 = lmp_date + pd.DateOffset(weeks=20)
    
    patient_conditions = htn_records[
        (htn_records['PERSON_ID'] == patient_id) & 
        (htn_records['CONDITION_START_DATETIME'] > lmp_date) &
        (htn_records['CONDITION_START_DATETIME'] < week_20)
    ]
    
    if not patient_conditions.empty:
        medical_history.at[idx, 'chtn_earlypreg'] = 1.0

n_early_htn = medical_history['chtn_earlypreg'].sum()
pct = n_early_htn / len(medical_history) * 100
print(f"  Found in {n_early_htn:,} pregnancies ({pct:.1f}%)\n")

print("Early pregnancy conditions extraction complete")
'''

# Save Medical History

In [ ]:
'''
# Save medical history
# note, prev file: df.to_pickle('pregnancy_cohort_hx_date.pkl')
OUTPUT_FILE = "pregnancy_cohort_hx_date.pkl" 
medical_history.to_pickle(OUTPUT_FILE)
print(f"\n Medical history saved to: {OUTPUT_FILE}")
'''